# State and input discretization: Q-learning

The minimum-energy problem of Unit 7, solved again without a model.

A double integrator must be driven from rest at the origin to position $1$
with zero velocity, minimising $\int_0^1 u(t)^2\,dt$. The analytical
solution is $z(t)=3t^2-2t^3$, $u(t)=6-12t$, with cost $12$. Unit 7
discretized state, input and time and solved the resulting finite grid
problem by exact dynamic programming, obtaining $16$ -- the discretization
penalty.

Here the *same grid problem* is solved by Q-learning. The grid dynamics
`M` and costs `C` are built below, but only so that the simulator can be
written; the learning algorithm sees them solely through calls to
`model_and_cost`.

Two features of this problem make it a good test. It is **deterministic**,
so the step size can be taken as $\gamma=1$: there is no sampling noise to
average away, and each update is exact. And it is a **finite horizon**
problem with a hard terminal constraint, so it uses the stage-indexed
driver `q_learning_finite_horizon`, with an infinite Q-factor assigned to
any action that fails to end at the target.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
plt.rcParams["figure.figsize"] = (14, 4)

from q_learning import q_learning_finite_horizon

rng = np.random.default_rng(0)

In [ ]:
# discretization parameters
dt = 0.5
K  = round(1 / dt)                 # number of stages
du = 0.2                           # input grid spacing
dv = dt * du                       # velocity grid spacing
dd = dt ** 2 / 2 * du              # position grid spacing

n_u = int(2 * (7 / du) + 1)        # inputs from -7 to +7
n_v = round(2 / dv)                # velocities from 0 to 2
n_d = round(1 / dd)                # positions from 0 to 1
n_x = (n_d + 1) * (n_v + 1)

print(f"{n_x} states, {n_u} inputs, {K} stages")

In [ ]:
# grid dynamics and costs, in index space
idx = np.arange(n_x).reshape(-1, 1)
ones_u = np.ones((1, n_u))
u_grid = np.arange(-(n_u - 1) // 2, (n_u - 1) // 2 + 1).reshape(1, -1)

D   = (idx // (n_v + 1)) @ ones_u          # position index
V   = (idx %  (n_v + 1)) @ ones_u          # velocity index
U   = np.ones((n_x, 1)) @ u_grid           # input index
V_1 = np.clip(V + U, 0, n_v)
D_1 = np.clip(D + 2 * V + U, 0, n_d)

M = (V_1 + (n_v + 1) * D_1).astype(int)    # next state
C = np.ones((n_x, 1)) @ (u_grid ** 2)      # stage cost

def model_and_cost(x, u):
    return int(M[x, u]), float(C[x, u])

terminal = n_d * (n_v + 1)                 # position 1, velocity 0

In [ ]:
Q, mu, J = q_learning_finite_horizon(
    np.full(n_x, n_u), model_and_cost,
    nsim=300_000, eps=0.99, alpha=1.0,
    prob_init_state=np.r_[1.0, np.zeros(n_x - 1)],
    horizon=K, terminal_index=terminal, rng=rng)

print(f"J[0,0] = {J[0, 0]}")
print(f"cost = dt * J[0,0] * du^2 = {dt * J[0, 0] * du ** 2:.4f}"
      f"   (Unit 7's exact grid DP gives 16)")

In [ ]:
# replay the learned policy from the initial state
d = np.zeros(K + 1); v = np.zeros(K + 1); u = np.zeros(K)
xbar = np.zeros(K + 1, dtype=int)
for k in range(K):
    ubar = mu[xbar[k], k]
    xbar[k + 1] = M[xbar[k], ubar]
    u[k] = (ubar - (n_u - 1) / 2) * du
    d[k + 1] = (xbar[k + 1] // (n_v + 1)) * dd
    v[k + 1] = (xbar[k + 1] %  (n_v + 1)) * dv
print("u =", u, "  position =", d, "  velocity =", v)

In [ ]:
# against the analytical continuous-time solution
t = np.linspace(0, 1, 1001)
z_sol, v_sol, u_sol = 3*t**2 - 2*t**3, 6*t - 6*t**2, 6 - 12*t
kvec = np.arange(K + 1) / K

fig, ax = plt.subplots(1, 3)
ax[0].stem(kvec[:-1], u); ax[0].plot(t, u_sol, "g"); ax[0].legend(["$u_k$", "$u(t)$"])
ax[1].stem(kvec, d);      ax[1].plot(t, z_sol, "g"); ax[1].legend(["$x_k$", "$x(t)$"])
ax[2].stem(kvec, v);      ax[2].plot(t, v_sol, "g"); ax[2].legend(["$v_k$", "$v(t)$"])
for a in ax: a.set_xlabel("time"); a.grid(True)
plt.tight_layout(); plt.show()

Q-learning recovers Unit 7's grid answer **exactly**: $u=[4,-4]$,
positions $[0,\,0.5,\,1]$, cost $16$.

That is a stronger statement than it looks. It is the same number that
exact dynamic programming produced in Unit 7 with full knowledge of the
grid dynamics -- obtained here by a method that only ever asked the
simulator "if I do this, what happens and what does it cost?".

The exactness comes from the problem being deterministic. With
$\gamma=1$ the update is a plain assignment, so once a state-action pair
has been tried at a stage, its Q-factor is the true one; no averaging and
no decaying step size are needed. In a stochastic problem -- the inventory
notebook next door -- neither holds, and the estimates only approach the
truth.

Only two stages are needed here, so the backward propagation of the
convergence proof happens almost immediately. Decreasing `dt` lengthens
the horizon, and the number of simulations required grows with it.